## **Sequential Patterns ( SPADE )**

Association rules ignore **time**: `{diapers, beer}` is the same in any order. In many problems the order is the information: *what do customers buy **after** a party? which product is bought **before** the next purchase of diapers?*
**Sequential pattern mining** finds ordered lists of itemsets, like `{chips} → {fruit}`, that occur in many **customers' histories**. Here it is done with **SPADE**, which is **Eclat for sequences**: a *vertical* id-list per item, joined by temporal intersections.

**Notebooks of this series:** 1. Apriori · 2. FP-Growth · 3. Eclat · 4. Comparing the Algorithms · 5. Association Rules ( Generation, Reduction and Evaluation ) · 6. Interestingness Measures · 7. Sequential Patterns ( SPADE ) · 8. High-Utility Itemsets · 9. Associative Classification ( CBA )

### **0. Environment Setup**

In [1]:
#Import Libs
import sklearn
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

In [2]:
#Define Colors Palette and Standards for Illustrations
RED, DARK, GRAY, LIGHT, SOFT = "#CD2027", "#404040", "#7F7F7F", "#D9D9D9", "#E9A0A3"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.2),
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titleweight": "bold",
    "axes.prop_cycle": cycler(color=[RED, DARK, SOFT, GRAY]),
})
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)

### **1. Dataset: Customers and Their Sequence of Visits**

The shopper profiles of the retail dataset ( cell below ) generate **each visit**. A **Markov chain over the profiles** generates the **order of the visits** of each customer, so that temporal patterns exist: after a *party* the customer tends to buy *healthy* products, the *baby* profile tends to repeat, *breakfast → family dinner → party → healthy → breakfast* is the most common cycle, and so on.

Attention! The transitions are **deliberately sharp** ( see the matrix below ) so that the temporal signal is visible in a small example. With flatter transitions the sequential rules barely beat a popularity baseline ( the evaluation in section 5 shows how to check it ).

In [4]:
#Generate the Transactions ( Shopper Profiles + Random Background Items )
rng = np.random.default_rng(RANDOM_STATE)
profiles = {
    "breakfast":     {"bread": .70, "butter": .50, "jam": .35, "milk": .60, "coffee": .50, "sugar": .30, "eggs": .40, "cereal": .30},
    "party":         {"beer": .60, "chips": .60, "soda": .50, "pizza": .35, "ice": .30, "peanuts": .30},
    "family dinner": {"pasta": .60, "tomato_sauce": .50, "cheese": .40, "chicken": .40, "rice": .40, "beans": .30, "salad": .30, "tomato": .30},
    "baby":          {"diapers": .80, "baby_food": .50, "wipes": .50, "milk": .40, "beer": .25},
    "healthy":       {"salad": .60, "fruit": .70, "yogurt": .50, "fish": .40, "lemon": .30, "water": .60, "tomato": .30},
}
items = sorted({i for p in profiles.values() for i in p} | {"baby_club_card"})

def make_basket():
    chosen = rng.choice(list(profiles), size=1 if rng.random() < 0.75 else 2, replace=False)
    basket = {i for p in chosen for i, prob in profiles[p].items() if rng.random() < prob}
    basket |= {i for i in items if rng.random() < 0.025}                      # random background items
    if "diapers" in basket:
        basket.add("baby_club_card")                                           # loyalty-card bundle (always together)
    return frozenset(basket) if basket else frozenset([str(rng.choice(items))])

transactions = [make_basket() for _ in range(6000)]
print("transactions:", len(transactions), "| distinct items:", len(items), "| mean basket size:", round(np.mean([len(t) for t in transactions]), 2))
for t in transactions[:3]:
    print("  ", sorted(t))

transactions: 6000 | distinct items: 31 | mean basket size: 4.72
   ['beans', 'fruit', 'pasta', 'tomato', 'water']
   ['fruit', 'salad', 'tomato_sauce']
   ['fish', 'fruit', 'salad', 'yogurt']


In [ ]:
#Simulate Customers: a Markov Chain over the Shopper Profiles Generates the Sequence of Visits
states = list(profiles)
transition = pd.DataFrame([[.05, .05, .65, .05, .20],      # from breakfast      ( usually followed by family dinner )
                           [.05, .05, .05, .05, .80],      # from party          ( usually followed by healthy )
                           [.10, .55, .10, .05, .20],      # from family dinner  ( usually followed by party )
                           [.05, .02, .05, .83, .05],      # from baby           ( usually repeats )
                           [.65, .15, .10, .05, .05]],     # from healthy        ( usually followed by breakfast )
                          index=states, columns=states)
start = np.array([.22, .22, .22, .12, .22])
rng_seq = np.random.default_rng(RANDOM_STATE)

def visit(profile):
    basket = {i for i, prob in profiles[profile].items() if rng_seq.random() < prob}
    basket |= {i for i in items if rng_seq.random() < 0.02}                      # random background items
    return frozenset(basket) if basket else frozenset([str(rng_seq.choice(items))])

def customer():
    state, visits = rng_seq.choice(states, p=start), []
    for _ in range(int(rng_seq.integers(3, 7))):                                  # 3 to 6 visits per customer
        visits.append(visit(state))
        state = rng_seq.choice(states, p=transition.loc[state].to_numpy())
    return visits

sequences = [customer() for _ in range(2000)]
fmt_visit = lambda v: "{" + ", ".join(sorted(v)) + "}"
print("customers:", len(sequences), "| mean visits:", round(np.mean([len(s) for s in sequences]), 2), "| mean items per visit:", round(np.mean([len(v) for s in sequences for v in s]), 2))
for s in sequences[:2]:
    print("  ", " -> ".join(fmt_visit(v) for v in s))
transition.round(2)

### **2. Sequences, Patterns and Support**

* A **sequence** is the ordered list of visits ( itemsets ) of one customer
* A **pattern** such as $\langle \{chips\}\ \{fruit\} \rangle$ is **contained** in a sequence if its itemsets appear **in that order** ( not necessarily in consecutive visits ), each one inside a visit
* **Support** = share of the **customers** ( not of the visits ) whose sequence contains the pattern

In [ ]:
#Containment and Support of a Pattern ( Greedy Leftmost Matching )
def contains(sequence, pattern):
    pos = 0
    for element in pattern:
        element = set(element)
        while pos < len(sequence) and not element <= sequence[pos]:
            pos += 1
        if pos == len(sequence):
            return False
        pos += 1
    return True

def sequence_support(pattern, data=sequences):
    return sum(contains(s, pattern) for s in data) / len(data)

fmt_pattern = lambda p: " -> ".join("{" + ", ".join(e) + "}" for e in p)
print("support of", fmt_pattern((("chips",), ("fruit",))), "=", round(sequence_support((("chips",), ("fruit",))), 3))

In [ ]:
#Order Matters: Support of "A then B" vs "B then A" ( if Time Were Irrelevant, Both Would Be Equal )
order_pairs = [("chips", "fruit"), ("beer", "salad"), ("soda", "water"), ("diapers", "baby_food"), ("coffee", "tomato")]
pd.DataFrame([{"A then B": f"{a} -> {c}", "support": sequence_support(((a,), (c,))), "B then A": f"{c} -> {a}", "support ": sequence_support(((c,), (a,)))} for a, c in order_pairs]).round(3)

### **3. SPADE: Vertical Id-Lists and Temporal Joins**

For each item the algorithm keeps an **id-list**: for each customer, the visits where the item occurs. A pattern grows by two kinds of joins:

* **i-extension:** add an item to the **last itemset** ( same visit ) → *intersect* the visits
* **s-extension:** add a **new itemset** in a *later* visit → keep the visits **after** the earliest end of the pattern

The candidates of a pattern are only the extensions that were frequent for its parent ( the Apriori property for sequences ).

In [ ]:
#SPADE Implementation ( Depth-First Search over Id-Lists with i- and s-Extensions )
from collections import defaultdict, Counter
from itertools import combinations
import time

def spade(sequences, min_support, max_length=4):
    n = len(sequences)
    min_count = int(np.ceil(min_support * n))
    occurrences = defaultdict(lambda: defaultdict(list))                          # item -> customer -> visits where it occurs
    for sid, seq in enumerate(sequences):
        for eid, element in enumerate(seq):
            for item in element:
                occurrences[item][sid].append(eid)
    atoms = {i: {sid: tuple(e) for sid, e in occ.items()} for i, occ in occurrences.items() if len(occ) >= min_count}
    frequent_items = sorted(atoms)
    result, stats = {}, {"joins": 0}

    def grow(pattern, idlist, length, candidates):
        result[pattern] = len(idlist) / n                                         # idlist: customer -> visits where the pattern can END
        if length == max_length:
            return
        last_item = pattern[-1][-1]
        freq_i, freq_s = {}, {}
        for item in candidates:
            if item > last_item:                                                  # i-extension ( alphabetical order avoids duplicates )
                new = {}
                for sid, ends in idlist.items():
                    common = tuple(e for e in ends if e in atoms[item].get(sid, ()))
                    if common:
                        new[sid] = common
                stats["joins"] += 1
                if len(new) >= min_count:
                    freq_i[item] = new
            new = {}                                                              # s-extension ( a later visit )
            for sid, ends in idlist.items():
                later = tuple(e for e in atoms[item].get(sid, ()) if e > ends[0])
                if later:
                    new[sid] = later
            stats["joins"] += 1
            if len(new) >= min_count:
                freq_s[item] = new
        next_candidates = sorted(set(freq_i) | set(freq_s))
        for item, new in freq_i.items():
            grow(pattern[:-1] + (pattern[-1] + (item,),), new, length + 1, next_candidates)
        for item, new in freq_s.items():
            grow(pattern + ((item,),), new, length + 1, next_candidates)

    for item in frequent_items:
        grow(((item,),), atoms[item], 1, frequent_items)
    return result, stats

t0 = time.perf_counter()
patterns, spade_stats = spade(sequences, min_support=0.10, max_length=4)
t_spade = time.perf_counter() - t0
pattern_table = pd.DataFrame({"pattern": [fmt_pattern(p) for p in patterns], "support": list(patterns.values()),
                              "items": [sum(len(e) for e in p) for p in patterns], "visits": [len(p) for p in patterns]})
print(f"sequential patterns ( support ≥ 10% of the customers ): {len(patterns)} | spanning more than one visit: {(pattern_table['visits'] > 1).sum()} | joins: {spade_stats['joins']} | time: {t_spade:.2f}s")
pattern_table[pattern_table["visits"] > 1].sort_values("support", ascending=False).head(10).round(3)

In [ ]:
#Number of Patterns by Length ( Items ) and by Number of Visits
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
pattern_table["items"].value_counts().sort_index().plot.bar(ax=axes[0], color=RED, width=0.7, rot=0); axes[0].set_xlabel("items in the pattern"); axes[0].set_ylabel("patterns"); axes[0].set_title("By number of items")
pattern_table["visits"].value_counts().sort_index().plot.bar(ax=axes[1], color=DARK, width=0.7, rot=0); axes[1].set_xlabel("visits spanned by the pattern"); axes[1].set_ylabel("patterns"); axes[1].set_title("By number of visits")
plt.tight_layout(); plt.show()

### **4. Quality Metrics**

In [ ]:
#Quality Check 1 ( Soundness ): Is the Support of Every Mined Pattern Correct? ( Recounted with contains() )
errors = [abs(sequence_support(p) - s) for p, s in patterns.items()]
print(f"patterns checked: {len(errors)} | maximum support error: {max(errors):.2e}")

In [ ]:
#Quality Check 2 ( Completeness ): Brute Force over Every Pattern with up to 2 Items ( Precision and Recall )
#Attention! The candidates are built from the frequent items: an infrequent item can never be part of a frequent pattern
single_items = [i for i in sorted(items) if sequence_support(((i,),)) >= 0.10]                      # frequent items, counted directly on the data
candidates = [((a,),) for a in single_items] + [((a,), (c,)) for a in single_items for c in single_items] + [((a, c),) for a, c in combinations(single_items, 2)]
truth = {p for p in candidates if sequence_support(p) >= 0.10}
found = {p for p in patterns if sum(len(e) for e in p) <= 2}
print(f"patterns with up to 2 items | found: {len(found)} | ground truth: {len(truth)} | precision: {len(found & truth) / len(found):.3f} | recall: {len(found & truth) / len(truth):.3f}")

### **5. Sequential Rules and Next-Visit Prediction**

A sequential rule `prefix → item` has **confidence** = support( prefix → item ) / support( prefix ): *among the customers who followed the prefix, how many bought the item later?*
The rules are mined on **70% of the customers** and used to predict the **last visit** of the other 30% from their previous visits ( **hit-rate@k**: the last basket contains at least one of the *k* recommended items ), against a **popularity baseline**.

In [ ]:
#Split the Customers and Mine the Sequential Rules on the Training Customers ( min support 4%, min confidence 20% )
perm = np.random.default_rng(RANDOM_STATE).permutation(len(sequences))
train_seq = [sequences[i] for i in perm[:1400]]
test_seq = [sequences[i] for i in perm[1400:]]

patterns_tr, _ = spade(train_seq, min_support=0.04, max_length=4)
rule_rows = [(p[:-1], p[-1][0], s, s / patterns_tr[p[:-1]]) for p, s in patterns_tr.items() if len(p) >= 2 and len(p[-1]) == 1 and p[:-1] in patterns_tr]
seq_rules = pd.DataFrame(rule_rows, columns=["antecedent", "consequent", "support", "confidence"])
seq_rules = seq_rules[seq_rules["confidence"] >= 0.20]
print("train customers:", len(train_seq), "| test customers:", len(test_seq), "| patterns:", len(patterns_tr), "| sequential rules:", len(seq_rules))
top_rules = seq_rules.sort_values("confidence", ascending=False).head(8)
top_rules.assign(antecedent=top_rules["antecedent"].map(fmt_pattern), consequent=top_rules["consequent"]).round(3)

In [ ]:
#Next-Visit Prediction: Rules vs Popularity vs "Repeat the Last Visit", Evaluated on the Unseen Customers
popularity = [i for i, _ in Counter(item for s in train_seq for v in s for item in v).most_common()]
rules_by_antecedent = defaultdict(list)
for r in seq_rules.itertuples():
    rules_by_antecedent[r.antecedent].append((r.consequent, r.confidence))

def evaluate_next_visit(skip_top=0, ks=(1, 3, 5)):
    skip = set(popularity[:skip_top])                                             # the most popular items are removed from recommendations AND targets
    pop_rank = [i for i in popularity if i not in skip]
    hits = {name: dict.fromkeys(ks, 0) for name in ["sequential rules", "popularity baseline", "repeat last visit"]}
    n_eval = 0
    for s in test_seq:
        history, target = s[:-1], set(s[-1]) - skip
        if not target:
            continue
        score = defaultdict(float)
        for antecedent, outcomes in rules_by_antecedent.items():
            if contains(history, antecedent):
                for item, conf in outcomes:
                    if item not in skip:
                        score[item] = max(score[item], conf)
        by_rules = sorted(score, key=lambda i: -score[i]) + [i for i in pop_rank if i not in score]
        last = sorted(set(history[-1]) - skip)
        by_repeat = last + [i for i in pop_rank if i not in last]
        for k in ks:
            hits["sequential rules"][k] += bool(set(by_rules[:k]) & target)
            hits["popularity baseline"][k] += bool(set(pop_rank[:k]) & target)
            hits["repeat last visit"][k] += bool(set(by_repeat[:k]) & target)
        n_eval += 1
    out = pd.DataFrame({name: {k: v / n_eval for k, v in h.items()} for name, h in hits.items()})
    out.index.name = "k"
    return out, n_eval

hit_all, n_all = evaluate_next_visit(skip_top=0)
hit_tail, n_tail = evaluate_next_visit(skip_top=5)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
hit_all.plot.bar(ax=axes[0], color=[RED, GRAY, DARK], width=0.75, rot=0, legend=False); axes[0].set_title(f"All items ( {n_all} customers )"); axes[0].set_ylabel("hit-rate@k")
hit_tail.plot.bar(ax=axes[1], color=[RED, GRAY, DARK], width=0.75, rot=0); axes[1].set_title(f"Without the 5 most popular items ( {n_tail} customers )")
plt.tight_layout(); plt.show()
print(f"hit-rate@k = share of the customers whose last visit contains at least one of the k recommended items | popular items removed in the right panel: {popularity[:5]}")
pd.concat({"all items": hit_all, "without top-5 popular": hit_tail}, axis=1).round(3)

Attention! The confidence of the rule means *"bought the item in a **later** visit"*, which is an approximation of the **next** visit. **Hit-rate@k** is easy to inflate with popular items ( milk, bread, salad appear in almost every profile ), so the right panel removes the 5 most popular items from the recommendations and from the targets, for **all** methods. A model is only useful if it beats **both** baselines: *popularity* and *repeat the last visit*.

### **6. Sensitivity to the Minimum Support**

In [ ]:
#Patterns, Joins and Time by Minimum Support
rows = []
for ms in [0.30, 0.20, 0.15, 0.10, 0.07, 0.05]:
    t0 = time.perf_counter(); res, st = spade(sequences, ms); elapsed = time.perf_counter() - t0
    rows.append({"min_support": ms, "patterns": len(res), "spanning > 1 visit": sum(len(p) > 1 for p in res), "joins": st["joins"], "time (s)": elapsed})
pd.DataFrame(rows).round(3)

### **7. Takeaways**

* **Order carries information:** compare the two supports of the *order matters* table; an itemset or an association rule cannot see it
* **Support is counted per customer**, not per visit, so frequent items quickly generate many patterns: use a higher support or a maximum length
* **SPADE is Eclat for sequences:** vertical id-lists and joins, depth-first, with the same independence between prefix classes
* **Evaluate on unseen customers** and always against baselines: with all items, the rules beat *popularity* at k = 1 and k = 5 and tie at k = 3; the gap is clear once the trivially popular items are removed. With flatter transitions the advantage disappears
* Related algorithms: **PrefixSpan** ( pattern growth over projected databases ) and **GSP** ( level-wise, like Apriori ); constraints such as maximum gap or window are common extensions